In [15]:
import os
import pandas as pd

In [16]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/Colab Notebooks/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cleaned_zone_df = pd.read_csv(os.path.join(file_path, 'cleaned zone dataset.csv'))
cleaned_zone_df.head()

,ftag,fthg,hometeam,season,awayteam,date,partition_3,partition_4
0,2,2,Aston Villa,1112,QPR,1,2,12
1,5,0,Wolves,1112,Man United,18,3,12
2,0,1,Liverpool,1213,QPR,19,5,13
3,0,2,Stoke,1112,Fulham,15,10,11
4,0,1,Everton,1213,Fulham,27,4,13


In [ ]:
cleaned_zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ftag         3800 non-null   int64 
 1   fthg         3800 non-null   int64 
 2   hometeam     3800 non-null   object
 3   season       3800 non-null   object
 4   awayteam     3800 non-null   object
 5   date         3800 non-null   int64 
 6   partition_3  3800 non-null   int64 
 7   partition_4  3800 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 237.6+ KB


In [ ]:
cleaned_zone_df.isnull().sum()

,0
ftag,0
fthg,0
hometeam,0
season,0
awayteam,0
date,0
partition_3,0
partition_4,0


In [19]:
df_home = cleaned_zone_df.rename(columns={
    "hometeam": "team",
    "fthg": "gols_feitos",
    "ftag": "gols_sofridos"
}).assign(tipo_jogo="home")

df_away = cleaned_zone_df.rename(columns={
    "awayteam": "team",
    "ftag": "gols_feitos",
    "fthg": "gols_sofridos"
}).assign(tipo_jogo="away")

# Junta home + away empilhando as partidas
df_long = pd.concat([df_home, df_away], ignore_index=True)

# Agora agrupa por time + temporada
df_final = (
    df_long.groupby(["team", "season"])
    .agg(
        total_gols_feitos=("gols_feitos", "sum"),
        total_gols_sofridos=("gols_sofridos", "sum"),
        total_partidas=("team", "count"),
        media_gols_por_partida=("gols_feitos", "mean"),
        media_gols_home=("gols_feitos", lambda x: x[df_long.loc[x.index, "tipo_jogo"] == "home"].mean()),
        media_gols_away=("gols_feitos", lambda x: x[df_long.loc[x.index, "tipo_jogo"] == "away"].mean()),
    )
    .reset_index()
)

df_final

,team,season,total_gols_feitos,total_gols_sofridos,total_partidas,media_gols_por_partida,media_gols_home,media_gols_away
0,Arsenal,1011,72,43,38,1.894737,1.736842,2.052632
1,Arsenal,1112,74,49,38,1.947368,2.052632,1.842105
2,Arsenal,1213,72,37,38,1.894737,2.473684,1.315789
3,Arsenal,1314,68,41,38,1.789474,1.894737,1.684211
4,Arsenal,1415,71,36,38,1.868421,2.157895,1.578947
...,...,...,...,...,...,...,...,...
195,Wigan,S910,37,79,38,0.973684,1.000000,0.947368
196,Wolves,1011,46,66,38,1.210526,1.578947,0.842105
197,Wolves,1112,40,82,38,1.052632,1.000000,1.105263
198,Wolves,1819,47,46,38,1.236842,1.473684,1.000000


In [21]:
output_path = file_path + 'curated_zone_etl.csv'
df_final.to_csv(output_path, index=False)